# 04 - Decoding & evaluation (Colab driver)

Thin driver: load a trained checkpoint -> **greedy-decode** the test set -> report **BLEU + chrF++ + TER**.

**Before running:**
- GPU runtime (Runtime -> Change runtime type -> GPU).
- `01_data` must have produced the cleaned eval files + tokenizer on Drive under `DATA_ROOT` (`test.clean.hi/.en`, `dev.clean.hi/.en`, `spm_unigram_16000.model`).
- Point `CKPT_PATH` at a trained `best.pt`. The smoke run wrote to local `/content` (wiped between sessions), so for a real eval save your run's `checkpoints/` to Drive and set the path.

*Greedy only for now; beam / MBR / ensemble and the manual-eval dump come later.*

## 1. Repo + install

In [1]:
import os, sys

REPO_URL = "https://github.com/Rishi-Jain-27/hindi-translator.git"
REPO_DIR = "/content/hindi-translator"

!test -d {REPO_DIR} && git -C {REPO_DIR} pull || git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

Already up to date.
cwd: /content/hindi-translator


In [2]:
!pip install -e .

Obtaining file:///content/hindi-translator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nmt (pyproject.toml) ... done
  Created wheel for nmt: filename=nmt-0.1.0-0.editable-py3-none-any.whl size=1279 sha256=bccf15dc7c6963c91b018231011fe14572ee51e49d8dcfd6ac3dbe5d28725cde
  Stored in directory: /tmp/pip-ephem-wheel-cache-tv34p3be/wheels/bd/c2/ef/ae818ff943d77ea8d63ef07aea61a1b82808716362dc169d4c
Successfully built nmt
  Attempting uninstall: nmt
    Found existing installation: nmt 0.1.0
    Uninstalling nmt-0.1.0:
      Successfully uninstalled nmt-0.1.0


In [3]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

device: cuda NVIDIA A100-SXM4-80GB


## 2. Config + Drive

`ModelConfig` must match what was trained. `DecodeConfig` drives greedy decoding (batch size + max generated length).

In [7]:
from nmt.config import DataConfig, ModelConfig, DecodeConfig
from google.colab import drive

# same Drive folder 01_data wrote the corpus + tokenizer to
drive.mount("/content/drive")
DATA_ROOT = "/content/drive/MyDrive/hindi-translator/data"

data_cfg = DataConfig(raw_dir=f"{DATA_ROOT}/raw", cache_dir=f"{DATA_ROOT}/cache")
model_cfg = ModelConfig()                        # MUST match the trained model (base: 512 / 8 heads / 6+6)
decode_cfg = DecodeConfig(
    mode="greedy",                               # greedy only for now (translate ignores other modes)
    batch_size=64,                               # sentences per decode batch; lower if you OOM
    max_decode_len=128,                          # hard cap on generated length; raise if outputs look cut off
)

# point at a TRAINED best.pt. the smoke run saved to local /content (wiped on reset);
# for a real eval, save your run's checkpoints/ to Drive and set this path.
CKPT_PATH = "/content/drive/MyDrive/hindi-translator/checkpoints/best.pt"
SPLIT = "test"                                   # "test" (~2507 pairs) or "dev" (~520, faster smoke)
print("ckpt:", CKPT_PATH, "| split:", SPLIT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ckpt: /content/hindi-translator/checkpoints/best.pt | split: dev


## 3. Tokenizer + model

In [8]:
from nmt.data.tokenizer import Tokenizer
from nmt.model.transformer import build_model

cache = data_cfg.cache_dir
tok = Tokenizer.load(os.path.join(cache, f"spm_{data_cfg.tokenizer_model}_{data_cfg.vocab_size}.model"))
print("tokenizer vocab:", tok.vocab_size)

model = build_model(model_cfg, tok).to(device)   # copies vocab_size + special ids from the tokenizer
print("params:", f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M")

tokenizer vocab: 16000
params: 52.3M


## 4. Load trained weights (EMA)

Load `best.pt`, then overlay the **EMA shadow** weights - those are what the training loop measured dev loss on (the raw weights jitter). Falls back to raw weights if the checkpoint has no EMA.

In [9]:
from nmt.train.ema import EMA

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])             # raw weights, incl. buffers (e.g. positional encodings)

if ckpt.get("ema"):
    ema = EMA(model)
    ema.load_state_dict(ckpt["ema"])
    ema.copy_to(model)                           # overlay the EMA params onto the model
    print("applied EMA weights")
else:
    print("no EMA in checkpoint -> using raw weights")

model.eval()
print("checkpoint step:", ckpt["step"], "| best dev nll:", ckpt.get("best"))

applied EMA weights
checkpoint step: 300 | best dev nll: 7.817545739216903


## 5. Read the eval set

Two line-aligned cleaned files -> source + reference lists. `read_lines` drops the trailing-newline empty and raises if the two files disagree in length.

In [10]:
from nmt.eval.evaluate import read_lines

src_path = os.path.join(cache, f"{SPLIT}.clean.hi")
ref_path = os.path.join(cache, f"{SPLIT}.clean.en")
srcs, refs = read_lines(src_path, ref_path)
print(f"{SPLIT}: {len(srcs)} sentence pairs")

dev: 520 sentence pairs


## 6. Translate + score

Greedy-decode every source and score against the references. **Note:** there's no KV-cache yet, so decoding recomputes the whole prefix each step; an undertrained model that rarely emits EOS will run to `max_decode_len` on every sentence and be slow. On a smoke checkpoint expect BLEU near 0 - the point here is that the pipeline runs end to end.

In [11]:
from nmt.eval.evaluate import evaluate

results = evaluate(model, tok, srcs, refs, decode_cfg)
results

bleu: 0.0017639756994600817
chrF: 1.6846661370170852
TER: 712.7578718783931


{'bleu': 0.0017639756994600817,
 'chrF': 1.6846661370170852,
 'TER': 712.7578718783931}

## 7. Eyeball a few translations

Source / model output / reference for the first few sentences - a quick sanity check beyond the aggregate numbers.

In [12]:
from nmt.decode.translate import translate

sample_src = srcs[:10]
sample_hyp = translate(model, sample_src, tok, decode_cfg)
for s, h, r in zip(sample_src, sample_hyp, refs[:10]):
    print("SRC:", s)
    print("HYP:", h)
    print("REF:", r)
    print("-" * 70)

SRC: महानगर पालिका अंतर्गत दत्तात्रय नगर माध्यमिक स्कूल के विद्यार्थियों ने काल्पनिक किला 'दत्तगढ़' बनाकर अपनी कल्पनाशक्ति का परिचय दिया।
HYP: The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The
REF: Students of the Dattatreya city Municipal corporation secondary school demonstrated their imagination power by creating the fictitious fort "Duttgarh".
----------------------------------------------------------------------
SRC: प्रधानाध्यापक संध्या मेडपल्लीवार के प्रोत्साहित करने पर शिक्षकों व विद्यार्थियों ने मिट्टïी से किले का निर्माण क